# PHASE 1 — NER batch 100 file (KHÔNG LLM)

Chạy KB + luật cho toàn bộ file, ghi:
- `phase1/<id>.json` — thực thể **chắc chắn** (luật xét nghiệm, từ điển thuốc, cascade KB tier1/2)
- `todo_llm/<id>.json` — việc **mờ** để Phase 2 định giá bằng LLM (span chưa quyết được type, ứng viên xét nghiệm vùng mờ)

**Nguyên tắc metric (sai type bị tính 2 lần, đều 0 điểm; trích thừa phạt ×3):**
**thà bỏ sót còn hơn làm thừa.** Span SYM_DIS mà KB **không quyết chắc** type
KHÔNG được gán bừa — đẩy sang Phase 2. Encoder thử **GPU** (không có vLLM ở
phase này), tự **fallback CPU** nếu nvrtc lỗi.

Settings: GPU T4 · Internet ON

In [ ]:
# Cell 1 — env tắt JIT (né nvrtc) TRƯỚC import torch, cài pyvi
import os
os.environ['PYTORCH_JIT'] = '0'                 # tắt TorchScript JIT
os.environ['PYTORCH_TENSOREXPR_FALLBACK'] = '2'
import sys, glob, json, re, subprocess, time
subprocess.run([sys.executable,'-m','pip','install','-q','pyvi'])
print('pyvi cài xong')

In [ ]:
# Cell 2 — dò model + code + KB + danh sách file
def find_dir(name):
    h = [x for x in glob.glob(f'/kaggle/input/**/{name}', recursive=True) if os.path.isdir(x)]
    return h[0] if h else None

MODEL_DIR = find_dir('ner_encoder')
assert MODEL_DIR and os.path.exists(f'{MODEL_DIR}/model.safetensors'), 'thiếu ner_encoder'
if not os.path.exists('fakeer'):
    subprocess.run(['git','clone','-q','https://github.com/Khanhhh239/fakeer'])
ROOT = 'fakeer' if os.path.exists('fakeer/src') else os.path.dirname(os.path.dirname(find_dir('src') or ''))
sys.path.insert(0, f'{ROOT}/src')
def kb(n):
    p = glob.glob(f'{ROOT}/kb/{n}') or glob.glob(f'/kaggle/input/**/kb/{n}', recursive=True)
    assert p, f'thiếu KB {n}'; return p[0]
ICD, RXN, INN = kb('icd10_vi_full.csv'), kb('rxnorm_merged.csv'), kb('inn_usan.csv')

# thư mục 100 file .txt: ưu tiên input/ trong dataset
IN = find_dir('input') or (f'{ROOT}/input' if os.path.exists(f'{ROOT}/input') else None)
FILES = sorted(glob.glob(f'{IN}/*.txt'), key=lambda x: int(re.sub(r'\D','',os.path.basename(x)) or 0)) if IN else []
assert FILES, 'không thấy file .txt đầu vào'
os.makedirs('/kaggle/working/phase1', exist_ok=True)
os.makedirs('/kaggle/working/todo_llm', exist_ok=True)
print(f'MODEL={MODEL_DIR}\nROOT={ROOT}\n{len(FILES)} file đầu vào')

In [ ]:
# Cell 3 — nạp encoder (thử GPU, fallback CPU) + cache embedding ICD ra .npy
import torch, numpy as np
for _f in ('_jit_set_texpr_fuser_enabled','_jit_set_nvfuser_enabled'):
    try: getattr(torch._C, _f)(False)
    except Exception: pass
try: torch._C._jit_override_can_fuse_on_gpu(False)
except Exception: pass

from transformers import AutoTokenizer, AutoModelForTokenClassification
tok = AutoTokenizer.from_pretrained(MODEL_DIR)
mdl = AutoModelForTokenClassification.from_pretrained(MODEL_DIR).eval()
ID2LAB = {int(k): v for k, v in mdl.config.id2label.items()}

# thử 1 forward trên GPU; nvrtc lỗi -> CPU
DEV = 'cpu'
if torch.cuda.is_available():
    try:
        mdl.to('cuda')
        _e = tok(['test'], is_split_into_words=True, return_tensors='pt', truncation=True, max_length=8).to('cuda')
        with torch.no_grad(): mdl(**_e)
        DEV = 'cuda'
    except Exception as _ex:
        print('encoder GPU lỗi -> CPU:', type(_ex).__name__, str(_ex)[:80]); mdl.to('cpu'); DEV = 'cpu'
print('encoder device:', DEV)

# --- ICD embedding: cache .npy để không encode lại 14.6k tên mỗi lần ---
import pandas as pd
_df = pd.read_csv(ICD)
_col = 'term' if 'term' in _df.columns else _df.columns[1]
ICD_CODES = _df['code'].astype(str).tolist()
ICD_NAMES = _df[_col].astype(str).tolist()
CACHE = '/kaggle/working/icd_emb.npy'
INP_CACHE = next(iter(glob.glob('/kaggle/input/**/icd_emb.npy', recursive=True)), None)

from transformers import AutoModel
EMB_NAME = 'AITeamVN/Vietnamese_Embedding'
etok = AutoTokenizer.from_pretrained(EMB_NAME)
emdl = AutoModel.from_pretrained(EMB_NAME).eval().to(DEV)
def embed(texts, bs=64):
    out = []
    with torch.no_grad():
        for i in range(0, len(texts), bs):
            b = texts[i:i+bs]
            enc = etok(b, padding=True, truncation=True, max_length=64, return_tensors='pt').to(DEV)
            h = emdl(**enc).last_hidden_state
            m = enc['attention_mask'].unsqueeze(-1).float()
            e = (h*m).sum(1) / m.sum(1).clamp(min=1e-9)
            e = torch.nn.functional.normalize(e, p=2, dim=1)
            out.append(e.cpu().numpy())
    return np.vstack(out).astype('float32')

if INP_CACHE:
    ICD_EMB = np.load(INP_CACHE); print('nạp ICD embedding từ cache dataset')
else:
    t0 = time.time(); ICD_EMB = embed(ICD_NAMES); np.save(CACHE, ICD_EMB)
    print(f'encode {len(ICD_NAMES)} tên ICD trong {time.time()-t0:.0f}s -> lưu {CACHE} (dùng lại lần sau)')
R_MASK = np.array([c.startswith('R') for c in ICD_CODES])
print('ICD_EMB', ICD_EMB.shape)

In [ ]:
# Cell 4 — hàm 3 nhánh (KB/luật). Encoder span + cascade tier1/2 (KB) ; tier3 -> để LLM.
from branch_b_lab_tests import extract_lab_pairs, lab_va_candidates
from branch_c_drugs import DrugMatcher
from utils.text_alignment import segment_with_map
from span_candidates import gen_candidates_for_regions, unresolved_regions
DRUG = DrugMatcher(RXN, INN)
TAU_KB = 0.93   # tier2: cosine >= -> CHẨN_ĐOÁN chắc

def encoder_spans(text):
    words, spans, ok = segment_with_map(text)
    if not ok or not words: return []
    W, S = 120, 100; best_lab = [None]*len(words); best_c = [-1]*len(words); i = 0
    while i < len(words):
        chunk = words[i:i+W]
        enc = tok(chunk, is_split_into_words=True, return_tensors='pt', truncation=True, max_length=256).to(DEV)
        with torch.no_grad(): pred = mdl(**enc).logits.argmax(-1)[0].cpu().numpy()
        wids = enc.word_ids(0); prev = None
        for pos, wid in enumerate(wids):
            if wid is None or wid == prev: prev = wid; continue
            prev = wid; gi = i+wid; c = min(wid, len(chunk)-1-wid)
            if c > best_c[gi]: best_c[gi] = c; best_lab[gi] = ID2LAB[int(pred[pos])]
        if i+W >= len(words): break
        i += S
    out, a = [], None
    def close(b):
        if a is None: return
        s, e = spans[a][0], spans[b][1]; out.append({'text': text[s:e], 'start': s, 'end': e})
    for k, lab in enumerate((best_lab or [])+['O']):
        lab = lab or 'O'
        if lab == 'B-SYM_DIS':
            if a is not None: close(k-1)
            a = k
        elif lab == 'I-SYM_DIS':
            if a is None: a = k
        else:
            if a is not None: close(k-1); a = None
    return out

def cascade_kb(span_text):
    """tier1: khớp chương R -> TRIỆU_CHỨNG. tier2: ngoài R & cos>=TAU -> CHẨN_ĐOÁN.
       không quyết được -> None (đẩy sang LLM). THÀ BỎ hơn gán bừa."""
    q = embed([span_text])[0]; sims = ICD_EMB @ q
    r_top = sims[R_MASK].max() if R_MASK.any() else -1
    o_top = sims[~R_MASK].max() if (~R_MASK).any() else -1
    if r_top > 0.5 and r_top >= o_top: return 'TRIỆU_CHỨNG'
    if o_top >= TAU_KB: return 'CHẨN_ĐOÁN'
    return None

In [ ]:
# Cell 5 — VÒNG LẶP 100 file
from utils.overlap_resolver import select_non_overlapping
from negation_detector import NegationDetector
NEG = NegationDetector()
t_all = time.time(); n_sure = n_todo = 0
for fp in FILES:
    fid = os.path.splitext(os.path.basename(fp))[0]
    TEXT = open(fp, encoding='utf-8').read()
    ents, todo = [], {'text_len': len(TEXT), 'sym_undecided': [], 'lab_candidates': []}

    # nhánh B (luật) + C (từ điển) — CHẮC
    for e in extract_lab_pairs(TEXT): ents.append({**e})
    for e in DRUG.extract_drugs(TEXT): ents.append({**e})

    # nhánh A: encoder -> span, cascade KB. Quyết được -> ents ; không -> todo (LLM)
    for s in encoder_spans(TEXT):
        lab = cascade_kb(s['text'])
        if lab: ents.append({**s, 'type': lab, 'score': 0.9, 'source': 'encoder+kb'})
        else:   todo['sym_undecided'].append(s)     # THÀ BỎ ở phase1, LLM quyết phase2

    # ứng viên xét nghiệm vùng mờ (luật bỏ sót) -> LLM phase2
    for c in lab_va_candidates(TEXT, [e for e in ents if e.get('type','').endswith('XÉT_NGHIỆM')]):
        todo['lab_candidates'].append(c)

    # khử chồng lấn cho phần CHẮC
    for e in ents: e.setdefault('score', 1.0); e.setdefault('source', 'rule')
    kept = select_non_overlapping(ents)
    kept = NEG.annotate_negation(kept, TEXT)
    for e in kept: e['assertion'] = NEG.get_assertion_status(e)
    kept.sort(key=lambda x: x['start'])
    res = {'text': TEXT, 'entities': [
        {'text': e['text'], 'type': e['type'], 'start': e['start'], 'end': e['end'],
         'score': round(float(e.get('score',1.0)),3), 'source': e.get('source','rule'),
         'negated': bool(e.get('negated', False)), 'assertion': e.get('assertion','affirmed')}
        for e in kept]}
    assert all(x['text'] == TEXT[x['start']:x['end']] for x in res['entities']), fid
    json.dump(res, open(f'/kaggle/working/phase1/{fid}.json','w',encoding='utf-8'), ensure_ascii=False)
    json.dump(todo, open(f'/kaggle/working/todo_llm/{fid}.json','w',encoding='utf-8'), ensure_ascii=False)
    n_sure += len(res['entities']); n_todo += len(todo['sym_undecided']) + len(todo['lab_candidates'])
print(f'\nXONG {len(FILES)} file trong {time.time()-t_all:.0f}s | {n_sure} thực thể chắc | {n_todo} việc cho LLM')
print('=> /kaggle/working/phase1/*.json  +  /kaggle/working/todo_llm/*.json  (+ icd_emb.npy)')